In [6]:
import os
import glob
from lxml import etree

# タクソノミ展開ディレクトリパス
TAXONOMY_DIR = './taxonomy'  # 解凍先パスに合わせて変更

# 名前空間定義（XSD定義用）
NS = {'xsd': 'http://www.w3.org/2001/XMLSchema'}

# XSDファイルを取得
xsd_files = glob.glob(os.path.join(TAXONOMY_DIR, '**/*.xsd'), recursive=True)

all_elements = []

for xsd_file in xsd_files:
    #print(f"Parsing {xsd_file}...")
    tree = etree.parse(xsd_file)
    root = tree.getroot()
    # xsd:elementタグを探索
    elements = root.findall('.//xsd:element', namespaces=NS)
    for el in elements:
        name = el.get('name')
        elem_type = el.get('type')
        abstract = el.get('abstract')
        substitution_group = el.get('substitutionGroup')

        all_elements.append({
            'file': xsd_file,
            'name': name,
            'type': elem_type,
            'abstract': abstract,
            'substitutionGroup': substitution_group
        })

# 結果を表示またはJSONなどで出力可能
for elem in all_elements[:10]:
    print(elem)

{'file': './taxonomy/jpctl/2021-11-01/jpctl_cor_2021-11-01.xsd', 'name': 'CabinetOfficeOrdinanceOnSystemForEnsuringAppropriatenessOfFinancialAndOtherDocumentsFormNo1InternalControlReportHeading', 'type': 'xbrli:stringItemType', 'abstract': 'true', 'substitutionGroup': 'iod:identifierItem'}
{'file': './taxonomy/jpctl/2021-11-01/jpctl_cor_2021-11-01.xsd', 'name': 'CoverPageHeading', 'type': 'xbrli:stringItemType', 'abstract': 'true', 'substitutionGroup': 'iod:identifierItem'}
{'file': './taxonomy/jpctl/2021-11-01/jpctl_cor_2021-11-01.xsd', 'name': 'BasicFrameworkOfInternalControlRelatedToFinancialReportingHeading', 'type': 'xbrli:stringItemType', 'abstract': 'true', 'substitutionGroup': 'iod:identifierItem'}
{'file': './taxonomy/jpctl/2021-11-01/jpctl_cor_2021-11-01.xsd', 'name': 'BasicFrameworkOfInternalControlRelatedToFinancialReportingTextBlock', 'type': 'nonnum:textBlockItemType', 'abstract': 'false', 'substitutionGroup': 'xbrli:item'}
{'file': './taxonomy/jpctl/2021-11-01/jpctl_cor_

In [8]:
import os
import glob
import json
import logging
from lxml import etree

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# EDINETタクソノミは以下のような名前空間を用いることが多い（年度やバージョンによって変更あり）
# 必要に応じて解析途中で実際に使われている名前空間を抽出して動的に対応してください。
NSMAP = {
    'xsd': 'http://www.w3.org/2001/XMLSchema',
    'link': 'http://www.xbrl.org/2003/linkbase',
    'xlink': 'http://www.w3.org/1999/xlink',
}

# 解析対象ディレクトリ（事前にタクソノミ一式を解凍して配置）
TAXONOMY_DIR = './taxonomy'

# 出力ファイル名
ELEMENTS_JSON = 'elements.json'
RELATIONSHIPS_JSON = 'relationships.json'

def parse_all_xsd_elements(taxonomy_dir):
    """
    タクソノミディレクトリ以下の全XSDファイルを走査し、定義済みの全要素を抽出する。
    戻り値: { element_name: { 'type':..., 'abstract':..., 'substitutionGroup':..., 'documentation':... }, ...}
    """
    elements = {}
    xsd_files = glob.glob(os.path.join(taxonomy_dir, '**/*.xsd'), recursive=True)
    logging.info(f"Found {len(xsd_files)} XSD files")

    for xf in xsd_files:
        try:
            tree = etree.parse(xf)
            root = tree.getroot()
            # XSD要素を取得
            for el in root.findall('.//xsd:element', namespaces=NSMAP):
                name = el.get('name')
                if name:
                    element_info = {
                        'type': el.get('type'),
                        'abstract': el.get('abstract'),
                        'substitutionGroup': el.get('substitutionGroup'),
                        'nillable': el.get('nillable'),
                        'documentation': ''
                    }
                    # documentation 取得（annotation内にある場合）
                    annotation = el.find('.//xsd:annotation/xsd:documentation', namespaces=NSMAP)
                    if annotation is not None:
                        element_info['documentation'] = annotation.text
                    elements[name] = element_info
        except Exception as e:
            logging.error(f"Error parsing XSD {xf}: {e}")

    return elements

def parse_linkbase_relationships(taxonomy_dir):
    """
    リンクベースファイル（.xmlなど）をパースし、要素間の関係を抽出する。
    プレゼンテーション、計算、定義、参照などのリンクベースを解析する。
    戻り値:
    {
      'presentation': [ { 'role':..., 'from':..., 'to':..., 'arcrole':..., 'order':..., ...}, ...],
      'calculation': [...],
      'definition': [...],
      'reference': [...]
    }
    """
    # リンクベースファイルのパターンは、.xmlや.linkなど様々なので、taxonomy直下にある.xmlを全て探索するとする
    link_files = glob.glob(os.path.join(taxonomy_dir, '**/*.xml'), recursive=True)
    logging.info(f"Found {len(link_files)} linkbase candidate files")

    relationships = {
        'presentation': [],
        'calculation': [],
        'definition': [],
        'reference': []
    }

    # arcroleに応じてどのカテゴリに入れるか判断する
    arcrole_mapping = {
        'http://www.xbrl.org/2003/arcrole/parent-child': 'presentation',
        'http://www.xbrl.org/2003/arcrole/summation-item': 'calculation',
        'http://www.xbrl.org/2003/arcrole/concept-definition': 'definition',
        'http://www.xbrl.org/2003/arcrole/concept-reference': 'reference'
    }

    for lf in link_files:
        try:
            tree = etree.parse(lf)
            root = tree.getroot()
            
            # link:roleType等が定義されている場合もあるが、ここではarcroleを直接みる
            arcs = root.findall('.//link:arc', namespaces=NSMAP)
            if not arcs:
                # EDINETでは, linkbaseArc要素が xlink:arcroleごとに '...'Arc という要素名で定義されることがある
                # 例えば presentationArc, calculationArc, definitionArc, referenceArc 等
                # これらを包括的に取得
                arcs = []
                for arc_tag_name in ['presentationArc', 'calculationArc', 'definitionArc', 'referenceArc']:
                    found = root.findall(f".//link:{arc_tag_name}", namespaces=NSMAP)
                    arcs.extend(found)

            for arc in arcs:
                arcrole = arc.get('{http://www.w3.org/1999/xlink}arcrole')
                from_ = arc.get('{http://www.w3.org/1999/xlink}from')
                to_ = arc.get('{http://www.w3.org/1999/xlink}to')
                order = arc.get('order')
                
                # locatorを介して実際の要素名を解決する
                from_elem = resolve_locator(root, from_)
                to_elem = resolve_locator(root, to_)

                if arcrole in arcrole_mapping:
                    rel_type = arcrole_mapping[arcrole]
                    relationships[rel_type].append({
                        'arcrole': arcrole,
                        'from': from_elem,
                        'to': to_elem,
                        'order': order,
                        'source_file': os.path.relpath(lf, taxonomy_dir)
                    })
        except Exception as e:
            # リンクベースでないファイルも混じる可能性があるのでログのみ
            logging.debug(f"Skipping {lf} due to parse error or no arcs: {e}")

    return relationships

def resolve_locator(root, loc_label):
    """
    linkbaseのfrom/to属性はlocタグで定義されたresourceを参照するlabelを用いる。
    この関数はlocatorを探して、hrefから要素名（qname）のローカル名を抽出する。
    """
    if loc_label is None:
        return None
    # locatorは: link:locタグでxlink:label属性でloc_labelを持ち、xlink:hrefに要素への参照がある
    loc = root.xpath(f".//link:loc[@xlink:label='{loc_label}']", namespaces=NSMAP)
    if not loc:
        # resourceの場合もあり得るが、ここではlocでなければNone返す
        return None
    loc = loc[0]
    href = loc.get('{http://www.w3.org/1999/xlink}href')
    # href例: something.xsd#ElementName
    if href and '#' in href:
        return href.split('#')[-1]
    return href  # fallback

if __name__ == "__main__":
    # 1. 全XSDを解析して要素一覧作成
    logging.info("Parsing XSD elements...")
    elements = parse_all_xsd_elements(TAXONOMY_DIR)
    logging.info(f"Extracted {len(elements)} elements")

    # 2. リンクベースを解析して要素間関係取得
    logging.info("Parsing linkbase relationships...")
    relationships = parse_linkbase_relationships(TAXONOMY_DIR)
    total_rel = sum(len(v) for v in relationships.values())
    logging.info(f"Extracted {total_rel} relationships in total")

    # 3. 結果をJSONで保存
    with open(ELEMENTS_JSON, 'w', encoding='utf-8') as f:
        json.dump(elements, f, indent=2, ensure_ascii=False)

    with open(RELATIONSHIPS_JSON, 'w', encoding='utf-8') as f:
        json.dump(relationships, f, indent=2, ensure_ascii=False)

    logging.info("Done. Elements and relationships exported.")

2024-12-14 23:12:36,351 [INFO] Parsing XSD elements...
2024-12-14 23:12:36,385 [INFO] Found 46 XSD files
2024-12-14 23:12:36,471 [INFO] Extracted 10698 elements
2024-12-14 23:12:36,472 [INFO] Parsing linkbase relationships...
2024-12-14 23:12:36,483 [INFO] Found 2542 linkbase candidate files


KeyboardInterrupt: 

In [16]:
import urllib
import re

def getStockCodeDataFrame() -> pd.core.frame.DataFrame:
    '''
    東証の上場銘柄コード一覧をdf形式で出力する。
    URL = 'https://www.jpx.co.jp/markets/statistics-equities/misc/01.html'
    '''

    URL = 'https://www.jpx.co.jp/markets/statistics-equities/misc/01.html'

    response = urllib.request.urlopen(URL).read().decode("utf-8")
    string_html = re.findall('<a href=\".+?\.xls\"', response)
    url_list = []
    for i in string_html:
        j = i.lstrip('<a href=\"')
        k = j.rstrip('\"')
        url_list.append('https://www.jpx.co.jp'+k)
    url = url_list[0]

    # 国内株式のみ抽出

    df = pd.read_excel(url)
    df = df[(df.iloc[:, 3] == 'プライム（内国株式）') | (df.iloc[:, 3] ==
                                               'スタンダード（内国株式）') | (df.iloc[:, 3] == 'グロース（内国株式）')]

    # 列名の変更
    df.columns = ['date', 'code', 'office_name', 'market_class', 'industry_detail_code',
                  'industry_detail', 'industry_code', 'industry', 'scale_code', 'scale_class']

    # 'date'列をdatetime->strへ型変換
    df['date'] = df['date'].astype(str)
    df['code'] = df['code'].astype(str)
    df['industry_detail_code'] = df['industry_detail_code'].astype(str)
    df['industry_code'] = df['industry_code'].astype(str)
    df['scale_code'] = df['scale_code'].astype(str)

    return df


In [18]:
import os
import json
import pandas as pd
import numpy as np
import requests
import yfinance as yf
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import shap

##############################################################
# 前段階: タクソノミ解析は別スクリプトで実行済みと想定
# elements.jsonとrelationships.jsonを利用する例
##############################################################
with open('elements.json', 'r', encoding='utf-8') as f:
    taxonomy_elements = json.load(f)
with open('relationships.json', 'r', encoding='utf-8') as f:
    taxonomy_relationships = json.load(f)

# taxonomy_elementsやtaxonomy_relationshipsはここでは参考程度
# 実務上はこれらを用いてXBRLから正しいタグを参照する際に使用可能。



In [ ]:
import requests
import zipfile
import io
from edinet_xbrl.edinet_xbrl_parser import EdinetXbrlParser

def download_xbrl(doc_id):
    """
    EDINETのdoc_idを用いてXBRLファイルを取得し、パースして主要財務データを返す関数。

    戻り値例:
    {
        'Revenue': float or None,
        'OperatingIncome': float or None,
        'NetIncome': float or None,
        'TotalAssets': float or None,
        'Equity': float or None,
        'EPS': float or None,
        'ROE': float or None,
        'ROA': float or None
    }
    """

    base_url = "https://disclosure.edinet-fsa.go.jp/api/v1/documents"
    params = {'type': 1}  # XBRLを含むZIPを取得するパラメータ
    r = requests.get(f"{base_url}/{doc_id}", params=params, stream=True)
    r.raise_for_status()  # HTTPエラー時は例外発生

    # ZIPファイルをメモリ上で解凍
    z = zipfile.ZipFile(io.BytesIO(r.content))
    # XBRLファイルを特定（.xbrlで終わるファイルを探す）
    xbrl_files = [f for f in z.namelist() if f.endswith('.xbrl')]
    if not xbrl_files:
        # XBRLファイルが見つからない場合
        raise FileNotFoundError("No XBRL file found in the downloaded ZIP.")

    xbrl_path = xbrl_files[0]
    with z.open(xbrl_path) as xf:
        parser = EdinetXbrlParser(xf)
        parser.parse()
        all_data = parser.to_dict()

    # 以下は、一般的な日本基準の有価証券報告書でよく見る財務項目（実際のタグは年度や企業による差異あり）
    # キー名はedinet_xbrlで標準化された名称を想定。該当要素が無い場合はNoneとなる。
    # タグ名は状況に合わせ修正が必要。

    # 売上高（Revenueに相当する項目: NetSalesやOperatingRevenue等が一般的）
    revenue_keys = ['NetSales', 'OperatingRevenue']  # 候補キー
    revenue = None
    for k in revenue_keys:
        if k in all_data:
            revenue = all_data.get(k)
            break

    # 営業利益
    operating_income_keys = ['OperatingIncome', 'OperatingProfit']
    operating_income = None
    for k in operating_income_keys:
        if k in all_data:
            operating_income = all_data.get(k)
            break

    # 純利益（親会社株主に帰属する当期純利益: ProfitLossAttributableToOwnersOfParentなど）
    net_income_keys = ['ProfitLossAttributableToOwnersOfParent', 'NetIncome']
    net_income = None
    for k in net_income_keys:
        if k in all_data:
            net_income = all_data.get(k)
            break

    # 総資産
    total_assets_keys = ['TotalAssets']
    total_assets = None
    for k in total_assets_keys:
        if k in all_data:
            total_assets = all_data.get(k)
            break

    # 自己資本（EquityAttributableToOwnersOfParentなど）
    equity_keys = ['EquityAttributableToOwnersOfParent', 'TotalEquity']
    equity = None
    for k in equity_keys:
        if k in all_data:
            equity = all_data.get(k)
            break

    # 1株当たり利益 (EPS)
    eps_keys = ['EarningsPerShare', 'BasicEarningsPerShare']
    eps = None
    for k in eps_keys:
        if k in all_data:
            eps = all_data.get(k)
            break

    # ROE, ROAはXBRL上に明確に存在しない場合もあるため、計算可能なら計算
    # ROE = NetIncome / Equity
    # ROA = NetIncome / TotalAssets
    # ここでは計算例として表示。実際はXBRLタグでROE/ROAが直接ある場合も。
    roe = None
    if net_income is not None and equity and equity != 0:
        roe = net_income / equity

    roa = None
    if net_income is not None and total_assets and total_assets != 0:
        roa = net_income / total_assets

    return {
        'Revenue': revenue,
        'OperatingIncome': operating_income,
        'NetIncome': net_income,
        'TotalAssets': total_assets,
        'Equity': equity,
        'EPS': eps,
        'ROE': roe,
        'ROA': roa
    }

# 使用例(実際には有効なdoc_idが必要):
# result = download_xbrl("S100OFFV")  # doc_idは適宜実際のIDに変更
# print(result)

In [ ]:
##############################################################
# EDINET APIを利用したXBRL取得&パース（概念例）
# （詳細なAPIパラメータやXBRLパースは省略し、疑似コード化）
##############################################################

def download_xbrl(doc_id):
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v1/documents"
    params = {'type': 1}
    r = requests.get(f"{base_url}/{doc_id}", params=params, stream=True)
    # ZIP解凍などは省略、XBRLパースも省略
    # parse_xbrlで必要な指標を抽出する想定
    return r.content

def parse_xbrl(xbrl_content, target_tags):
    # xbrl_contentから、必要な指標を抽出する仮関数
    # target_tags = ['Revenue', 'OperatingIncome', 'NetIncome', 'TotalAssets', 'Equity', 'EPS', 'ROE', 'ROA']
    # 実際にはXBRLパーサー利用
    financials = {
        'Revenue': np.random.uniform(1e9, 1e12),
        'OperatingIncome': np.random.uniform(1e8, 1e11),
        'NetIncome': np.random.uniform(1e7, 1e10),
        'TotalAssets': np.random.uniform(1e9, 1e12),
        'Equity': np.random.uniform(1e8, 1e11),
        'EPS': np.random.uniform(10, 2000),
        'ROE': np.random.uniform(0, 0.3),
        'ROA': np.random.uniform(0, 0.2)
    }
    return financials

##############################################################
# 全銘柄リスト取得（事前に用意）
# 例: tickers.csvに[ticker, market, name, ...]
##############################################################
tickers_df = getStockCodeDataFrame()
tickers = tickers_df['code'].tolist()

##############################################################
# 対象期間設定・データ取得方針
# 例: 2010-2015年末時点の財務情報を特徴量にして、
# 2015-2020年の株価成長率をラベルにする。
##############################################################
feature_start = '2010-01-01'
feature_end = '2015-12-31'
label_start = '2015-01-01'
label_end = '2020-12-31'

##############################################################
# 財務データ取得（概念例）
##############################################################
financial_records = []
for ticker in tickers:
    # EDINETコードにマッピング必要だがここは省略
    # doc_idリストも実際にはEDINET APIで期間指定して取得
    # ここでは疑似的にループで作成
    for year in range(2010, 2016):
        doc_id = f"FAKE-{ticker}-{year}"  # 擬似doc_id
        try:
            xbrl_zip = download_xbrl(doc_id)
            fin = parse_xbrl(xbrl_zip, target_tags=['Revenue','OperatingIncome','NetIncome','TotalAssets','Equity','EPS','ROE','ROA'])
            fin['ticker'] = ticker
            fin['fiscal_year'] = year
            financial_records.append(fin)
        except Exception as e:
            pass

financial_df = pd.DataFrame(financial_records)

# 成長率など、過去からの変化指標を計算
financial_df.sort_values(['ticker','fiscal_year'], inplace=True)
financial_df['RevenueGrowth'] = financial_df.groupby('ticker')['Revenue'].pct_change()
financial_df['EPSGrowth'] = financial_df.groupby('ticker')['EPS'].pct_change()
financial_df['ROEChange'] = financial_df.groupby('ticker')['ROE'].diff()
financial_df['ROAChange'] = financial_df.groupby('ticker')['ROA'].diff()

##############################################################
# 株価データ取得 (yfinance利用)
##############################################################
price_data = {}
for t in tickers:
    yf_ticker = f"{t}.T"  # 日本株なら.T必要
    try:
        p = yf.download(yf_ticker, start='2010-01-01', end='2020-12-31')
        price_data[t] = p
    except:
        continue

# 年次末株価取得
def get_year_end_price(df):
    # 年末（12月末）の株価を取得
    return df.resample('Y').last()['Close']

price_yearly = []
for t, p in price_data.items():
    py = get_year_end_price(p)
    # fiscal_yearと合わせるため、year列付与
    for y, val in zip(py.index.year, py.values):
        price_yearly.append({'ticker': t, 'year': y, 'close': val})
price_yearly_df = pd.DataFrame(price_yearly)

##############################################################

# 同様にy_10xについてもモデル学習し、10倍銘柄の特徴を分析可能。

In [ ]:
# ラベル作成
# 2015年の株価を基準に2020年までに何倍になったか
##############################################################
base_year = 2015
target_year = 2020

base_prices = price_yearly_df[price_yearly_df['year'] == base_year][['ticker','close']].rename(columns={'close':'base_close'})
target_prices = price_yearly_df[price_yearly_df['year'] == target_year][['ticker','close']].rename(columns={'close':'target_close'})
merged_price = pd.merge(base_prices, target_prices, on='ticker', how='inner')
merged_price['multiplier'] = merged_price['target_close'] / merged_price['base_close']
# 5倍以上を1、それ以下0
merged_price['label_5x'] = (merged_price['multiplier'] >= 5.0).astype(int)
merged_price['label_10x'] = (merged_price['multiplier'] >= 10.0).astype(int)

##############################################################
# 特徴量とラベル結合
# 2015年時点の財務指標で予測
##############################################################
feature_year = 2015
features_df = financial_df[financial_df['fiscal_year'] == feature_year].copy()
features_df = pd.merge(features_df, merged_price[['ticker','label_5x','label_10x']], on='ticker', how='inner')

X = features_df[['Revenue','OperatingIncome','NetIncome','TotalAssets','Equity','EPS','ROE','ROA','RevenueGrowth','EPSGrowth','ROEChange','ROAChange']].fillna(0)
y_5x = features_df['label_5x']
y_10x = features_df['label_10x']

##############################################################
# モデル学習（5倍銘柄判定）
##############################################################
# データが少ない場合はKFold、ある程度あればTimeSeriesSplitなどを使用
tscv = TimeSeriesSplit(n_splits=3)
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
aucs = []
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_5x.iloc[train_idx], y_5x.iloc[test_idx]
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_pred)
    aucs.append(auc)

print("5x classification AUC mean:", np.mean(aucs))

##############################################################
# 特徴量重要度
##############################################################
importances = model.feature_importances_
for f, imp in zip(X.columns, importances):
    print(f"{f}: {imp}")

##############################################################
# SHAPによる特徴量解釈
##############################################################
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values[1], X)

##############################################################
# 結果解釈
# AUCや特徴量重要度から、有用な財務指標、成長率指標などを確認。
# これにより5倍、10倍銘柄に共通する統計的特徴を特定可能。
##############################################################
